# Create `BED` File of Unique Splice Junctions

## Purpose: 

* Take all `rMATS SE` event files. 
* Convert each splice junction in file to genomic region in `BED` file.  
    * NOTE: `- strand` features are off by 1 base pair (in some cases) but that didn't matter for my use case. 
* Create and output final sorted `BED` file with unique IDs for each line. 

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import glob, os

## Literals

In [2]:
# get paths to all SE event rMATS files
SE_files = glob.glob(
    "/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/**/SE*.txt", 
    recursive=True
)
print(SE_files)

# columns with coordinate numbers 
exon_coordinate_columns = ["exonStart_0base", "exonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"]

# unique id information 
id_creation_columns = ["chr", "start", "strand"]

# final order of columns for BED file
final_bed_order = ["chr", "start", "end", "name", "score", "strand"]

bedtools_path = "/project/PlatigLab/software/bedtools-v2.31.1/bin/bedtools"

['/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/SUB1-BGKLV13-K562/SE.MATS.JC.txt', '/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/KHSRP-BGHLV12-HepG2/SE.MATS.JC.txt', '/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/YBX3-BGHLV20-HepG2/SE.MATS.JC.txt', '/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/WDR43-BGKLV34-K562/SE.MATS.JC.txt', '/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/DDX21-BGHLV14-HepG2/SE.MATS.JC.txt', '/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/RBM34-BGKLV11-K562/SE.MATS.JC.txt', '/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/NCBP2-BGKLV08-K562/SE.MATS.JC.txt', '/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/DDX59-BGHLV35-HepG2/SE.MAT

## Extract All `SE` Splice Junctions

#### **IMPORTANT NOTE:**

Technically this is incorrect in that for `- strand` features, I am defining the base pair that is one in front of the splice junction. 

However, for our purposes, this doesn't matter enough. 

In [3]:
# # key is "chr_start_stop_strand"
# # value is list with chr_start_stop_strand_
# all_splice_junctions = {}

all_splice_junctions = []

# for each SE file 
for SE_file in SE_files: 
    
    # load dataframe and subset to coordinate-relevant columns
    tmp_df = pd.read_csv(
        SE_file,
        sep="\t", 
    )
    
    # for each start coordinate feature 
    for start_coord in exon_coordinate_columns: 
        
        # just subset to the chromosome and strand 
        append_df = tmp_df[["chr", start_coord, "strand"]].copy()
        
        # add 1 to the start coordinate to create the end coordinate 
        # (see note in Markdown header above)
        append_df["end"] = append_df[start_coord] + 1
        
        # clearly mark the start coordinate 
        # ALSO NEEDED for the concat step as it uses column names 
        append_df = append_df.rename(columns={start_coord: "start"})
        
        # append this df to the all splice junctions list that will be 
        # used to create BED file of all unique splice junctions
        all_splice_junctions.append(append_df)


## Get Unique Splice Junctions

In [4]:
# concatenate all splice junction information and drop the duplicates 
unique_splice_junctions = pd.concat(all_splice_junctions).drop_duplicates()

# add needed columns for BED file 
unique_splice_junctions["name"] = unique_splice_junctions["chr"] + "_" + unique_splice_junctions["start"].astype(str) + "_" + unique_splice_junctions["strand"]
unique_splice_junctions["score"] = "."

# make sure that chromosome start coordinate is unique ID 
assert unique_splice_junctions["name"].is_unique

# order the columns per BED file specifications
unique_splice_junctions = unique_splice_junctions[final_bed_order]

unique_splice_junctions.head()

,chr,start,end,name,score,strand
0,chrX,156022698,156022699,chrX_156022698_+,.,+
1,chrX,155492356,155492357,chrX_155492356_-,.,-
2,chrX,155506897,155506898,chrX_155506897_-,.,-
3,chrX,155524455,155524456,chrX_155524455_-,.,-
4,chrX,155545095,155545096,chrX_155545095_-,.,-


## Output `BED` File

In [5]:
# output as BED file 
unique_splice_junctions.to_csv("../output/bedtools_input/unique_splicing_junctions.bed", sep="\t", header=False, index=False)